# Country Risk & Sovereign Stress Model
## Capital Markets Intelligence Platform

Sovereign rating impact scenarios and stress testing framework.

**Lens**: JPM Country Risk style

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Stress Test Results

In [ ]:
stress_df = pd.read_csv('../data/processed/stress_test_results.csv')
print(f'Loaded {len(stress_df)} scenario-country combinations')
stress_df.head()

## 2. Risk Heatmap by Scenario

In [ ]:
# Pivot for heatmap
heatmap_data = stress_df.pivot_table(index='country', columns='scenario', values='risk_score')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Risk Score'})
ax.set_title('Sovereign Risk Scores by Scenario', fontsize=14)
plt.tight_layout()
plt.savefig('../output/risk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Yield Impact Analysis

In [ ]:
# Yield changes under each scenario
stress_df['yield_change_bps'] = (stress_df['stressed_yield_pct'] - stress_df['ten_yr_yield_pct']) * 100

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
scenarios = [s for s in stress_df['scenario'].unique() if s != 'Base Case']

for ax, scenario in zip(axes.flatten(), scenarios):
    subset = stress_df[stress_df['scenario'] == scenario].sort_values('yield_change_bps')
    colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in subset['yield_change_bps']]
    ax.barh(subset['country'], subset['yield_change_bps'], color=colors)
    ax.set_title(scenario, fontsize=10)
    ax.set_xlabel('Yield Change (bps)')
    ax.axvline(x=0, color='black', linewidth=0.5)

plt.suptitle('10Y Yield Impact by Scenario', fontsize=14)
plt.tight_layout()
plt.savefig('../output/yield_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Most Vulnerable Sovereigns

In [ ]:
# Average risk score across all stress scenarios (excluding base case)
avg_risk = stress_df[stress_df['scenario'] != 'Base Case'].groupby('country')['risk_score'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
avg_risk.plot(kind='bar', ax=ax, color='#e67e22')
ax.set_title('Average Risk Score Across Stress Scenarios')
ax.set_ylabel('Avg Risk Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../output/vulnerability_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print('Vulnerability Ranking:')
for i, (country, score) in enumerate(avg_risk.items(), 1):
    print(f'  {i}. {country}: {score:.1f}')

## 5. Key Findings

- EM sovereigns (Turkey, Brazil, South Africa) most vulnerable across scenarios
- DM safe havens (US, Germany, Japan) benefit from flight-to-quality in recession
- Oil shock scenario creates divergent outcomes: exporters benefit, importers suffer
- EM currency crisis has the highest tail risk concentration

---
*Generated by Capital Markets Intelligence Platform*